# E2 — Bentong fixed S2 + R1 grouped OOF benchmark

This standalone notebook reconstructs the **current thesis E2 experiment**:

- source domain: Bentong;
- fixed predictors: 13 Sentinel-2 variables;
- fixed classifier: R1 RBF-SVC (`C=10`, `gamma='scale'`, `tol=0.001`);
- fixed evaluation: the five pre-existing Bentong outer folds;
- evaluation unit: pixel metrics plus polygon metrics aggregated by mean
  multiclass decision score.

E2 is a matched fixed-configuration grouped OOF benchmark for comparison
with E3. It does **not** repeat feature selection, tune hyperparameters,
replace E1 nested model development, or fit a final all-data model.

Historical internal label `Mixed agriculture` remains numerically encoded
as class 3. Reporting uses the thesis label `Other agriculture`; no target
values or predictions are changed.


## 1. Mount Drive and import libraries

Expected environment: Google Colab with scikit-learn 1.6.1. A different
library version is reported because solver-level differences can affect
exact reproduction.


In [ ]:
from pathlib import Path
import os


def _find_repository_root(start: Path) -> Path:
    current = start.resolve()
    while current.parent != current:
        if (current / "README.md").exists() and (current / "code").exists():
            return current
        current = current.parent
    raise RuntimeError("Run this notebook from within the repository tree.")


REPO_ROOT = _find_repository_root(Path.cwd())
DATA_ROOT = Path(
    os.environ.get("DURIAN_DATA_ROOT", REPO_ROOT / "private_data")
).expanduser().resolve()
REPO_OUTPUT_ROOT = Path(
    os.environ.get("DURIAN_OUTPUT_ROOT", REPO_ROOT / "outputs" / "runs")
).expanduser().resolve()
REPO_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Private data root:", DATA_ROOT)
print("Run output root:", REPO_OUTPUT_ROOT)


import gc
import hashlib
import json
import os
import platform
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

print('Python:', platform.python_version())
print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)
if sklearn.__version__ != '1.6.1':
    warnings.warn(
        'Original SVM workflow used scikit-learn 1.6.1. Exact score '
        'reproduction should be assessed with that version.',
        RuntimeWarning,
    )

## 2. Locked paths, class mapping and E2 configuration

Change only the three input paths if your Drive layout differs. Do not
change the rows, folds, feature order, weights or SVM parameters when
reproducing E2.


In [ ]:
INPUT_CSV = DATA_ROOT / r"Bentong_pixel_samples_2025_v1.csv"
TRAINING_ROWS_PATH = DATA_ROOT / r"RF_grouped_validation_20260623_153912_UTC/tables/training_rows_used.csv"
FOLD_ASSIGNMENTS_PATH = DATA_ROOT / r"RF_grouped_validation_20260623_153912_UTC/tables/fold_assignments.csv"

RUN_NAME = 'E2_Bentong_fixed_S2_R1_grouped_OOF_20260813_run01'
OUTPUT_DIR = REPO_OUTPUT_ROOT / RUN_NAME
TABLE_DIR = OUTPUT_DIR / 'tables'
METADATA_DIR = OUTPUT_DIR / 'metadata'
MODEL_DIR = OUTPUT_DIR / 'models_by_fold'
for directory in [TABLE_DIR, METADATA_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_SPLITS = 5
CSV_CHUNK_SIZE = 100_000
SVC_CACHE_MB = 1200
VERIFY_INPUT_CSV_SHA256 = True
SAVE_FOLD_MODELS = False
REQUIRE_LOCKED_RESULT_MATCH = True
RESULT_ABSOLUTE_TOLERANCE = 1e-10

EXPECTED_INPUT_CSV_SHA256 = (
    '39bea3aca9b202e7e9155dbc0a68eb04795709db91ecda7edd2650489acdb0fb'
)
EXPECTED_TRAINING_ROWS_SHA256 = (
    '0de3149cc9dd43fdccca6081db2d4939966896c585e5c61879740590987fdfef'
)
EXPECTED_FOLD_ASSIGNMENTS_SHA256 = (
    '6563feb8782fcfb57b13c42c7b0ff84fa0f49d3f3c7afc731fa7451d1d8e8a2f'
)
EXPECTED_MODEL_ROWS = 37_724
EXPECTED_SAMPLE_COUNT = 557
EXPECTED_GROUP_COUNT = 206

CLASS_TO_ID = {
    'Built-up/Bare soil': 0,
    'Durian': 1,
    'Forest': 2,
    'Mixed agriculture': 3,
    'Oil palm': 4,
    'Rubber': 5,
    'Water': 6,
}
ID_TO_INTERNAL_CLASS = {
    class_id: name for name, class_id in CLASS_TO_ID.items()
}
ID_TO_REPORTING_CLASS = {
    0: 'Built-up/Bare soil',
    1: 'Durian',
    2: 'Forest',
    3: 'Other agriculture',
    4: 'Oil palm',
    5: 'Rubber',
    6: 'Water',
}
CLASS_IDS = sorted(ID_TO_INTERNAL_CLASS)
REPORTING_CLASS_NAMES = [
    ID_TO_REPORTING_CLASS[class_id] for class_id in CLASS_IDS
]
DURIAN_ID = CLASS_TO_ID['Durian']
RUBBER_ID = CLASS_TO_ID['Rubber']

S2_FEATURES = [
    'B3', 'B4', 'B5', 'B6', 'B7',
    'B8', 'B8A', 'B11', 'B12',
    'NDVI', 'NDRE', 'NDWI', 'EVI',
]
R1_PARAMS = {
    'candidate_id': 'R1',
    'model_type': 'rbf_svc',
    'C': 10.0,
    'gamma': 'scale',
    'tol': 0.001,
}

missing_inputs = [
    str(path)
    for path in [INPUT_CSV, TRAINING_ROWS_PATH, FOLD_ASSIGNMENTS_PATH]
    if not path.exists()
]
if missing_inputs:
    raise FileNotFoundError(
        'Missing required input file(s):\n' + '\n'.join(missing_inputs)
    )

print('Output directory:', OUTPUT_DIR)
print('Fixed features:', S2_FEATURES)
print('Fixed R1 parameters:', R1_PARAMS)


## 3. Reproducibility and atomic-output helpers

Hash checks prevent a similarly named but different input table from
silently changing E2. Atomic writes avoid incomplete final files after
an interrupted Colab session.


In [ ]:
def sha256_file(path, chunk_size=16 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        while True:
            block = stream.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def atomic_write_csv(frame, path, **kwargs):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, **kwargs)
    os.replace(temporary, path)


def atomic_write_json(payload, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with open(temporary, 'w', encoding='utf-8') as stream:
        json.dump(payload, stream, indent=2, ensure_ascii=False)
    os.replace(temporary, path)


def json_value(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f'Not JSON serializable: {type(value).__name__}')


## 4. Load and audit the exact retained rows and fixed fold assignments

Required invariants: 37,724 pixel rows, 557 polygons, 206 groups, unique
`pixel_uid`, polygon weights summing to one, and no `group_uid` crossing
outer folds.


In [ ]:
training_rows_hash = sha256_file(TRAINING_ROWS_PATH)
fold_assignments_hash = sha256_file(FOLD_ASSIGNMENTS_PATH)
if training_rows_hash != EXPECTED_TRAINING_ROWS_SHA256:
    raise ValueError('training_rows_used.csv SHA256 mismatch.')
if fold_assignments_hash != EXPECTED_FOLD_ASSIGNMENTS_SHA256:
    raise ValueError('fold_assignments.csv SHA256 mismatch.')

training_rows = pd.read_csv(
    TRAINING_ROWS_PATH,
    dtype={
        'pixel_uid': 'string',
        'sample_uid': 'string',
        'group_uid': 'string',
        'class_lv2': 'string',
    },
)
fold_assignments = pd.read_csv(
    FOLD_ASSIGNMENTS_PATH,
    dtype={
        'sample_uid': 'string',
        'group_uid': 'string',
        'class_lv2': 'string',
    },
)
for column in ['class_id', 'fold_id']:
    training_rows[column] = pd.to_numeric(
        training_rows[column], errors='raise'
    ).astype('int64')
    fold_assignments[column] = pd.to_numeric(
        fold_assignments[column], errors='raise'
    ).astype('int64')
training_rows['sample_weight'] = pd.to_numeric(
    training_rows['sample_weight'], errors='raise'
).astype('float64')

if len(training_rows) != EXPECTED_MODEL_ROWS:
    raise ValueError('Unexpected retained pixel-row count.')
if training_rows['pixel_uid'].isna().any():
    raise ValueError('pixel_uid contains null values.')
if training_rows['pixel_uid'].duplicated().any():
    raise ValueError('pixel_uid is not unique.')
if training_rows['sample_uid'].nunique() != EXPECTED_SAMPLE_COUNT:
    raise ValueError('Unexpected polygon count.')
if training_rows['group_uid'].nunique() != EXPECTED_GROUP_COUNT:
    raise ValueError('Unexpected group count.')
if sorted(training_rows['class_id'].unique()) != CLASS_IDS:
    raise ValueError('The locked seven classes are not all present.')
if sorted(training_rows['fold_id'].unique()) != list(range(N_SPLITS)):
    raise ValueError('Unexpected fold IDs.')
if training_rows.groupby('group_uid')['fold_id'].nunique().max() != 1:
    raise ValueError('A spatial group crosses outer folds.')

polygon_weight_totals = training_rows.groupby('sample_uid')[
    'sample_weight'
].sum()
if not np.allclose(polygon_weight_totals.to_numpy(), 1.0):
    raise ValueError('Per-polygon pixel weights do not sum to one.')

sample_design = training_rows[
    ['sample_uid', 'group_uid', 'class_id', 'class_lv2', 'fold_id']
].drop_duplicates('sample_uid')
fold_check = sample_design.merge(
    fold_assignments,
    on='sample_uid',
    how='outer',
    suffixes=('_training', '_assignment'),
    indicator=True,
    validate='one_to_one',
)
if not (fold_check['_merge'] == 'both').all():
    raise ValueError('Training rows and fold assignments differ.')
for column in ['group_uid', 'class_id', 'class_lv2', 'fold_id']:
    if not (
        fold_check[f'{column}_training'].astype(str)
        == fold_check[f'{column}_assignment'].astype(str)
    ).all():
        raise ValueError(f'Fold metadata mismatch: {column}')

design_summary = (
    training_rows.groupby(['class_id', 'class_lv2'])
    .agg(
        pixel_rows=('pixel_uid', 'size'),
        polygons=('sample_uid', 'nunique'),
        groups=('group_uid', 'nunique'),
    )
    .reset_index()
    .sort_values('class_id')
)
display(design_summary)
print('Training rows SHA256:', training_rows_hash)
print('Fold assignments SHA256:', fold_assignments_hash)


## 5. Recover the 13 S2 predictors for the exact retained pixel IDs

The 712,772-row source CSV is scanned in chunks. Only the 37,724 locked
rows are retained, which keeps memory use stable and preserves the E1/E2
input design.


In [ ]:
if VERIFY_INPUT_CSV_SHA256:
    input_csv_hash = sha256_file(INPUT_CSV)
    if input_csv_hash != EXPECTED_INPUT_CSV_SHA256:
        raise ValueError('Bentong input CSV SHA256 mismatch.')
else:
    input_csv_hash = None
    warnings.warn('Large input CSV SHA256 verification was skipped.')

source_columns = ['pixel_uid'] + S2_FEATURES
source_header = pd.read_csv(INPUT_CSV, nrows=0).columns.tolist()
missing_columns = sorted(set(source_columns).difference(source_header))
if missing_columns:
    raise ValueError(f'Input CSV is missing: {missing_columns}')

selected_pixel_ids = set(training_rows['pixel_uid'].astype(str))
retained_parts = []
scanned_rows = 0
retained_count = 0

for chunk_index, chunk in enumerate(
    pd.read_csv(
        INPUT_CSV,
        usecols=source_columns,
        dtype={'pixel_uid': 'string'},
        chunksize=CSV_CHUNK_SIZE,
        low_memory=False,
    ),
    start=1,
):
    scanned_rows += len(chunk)
    keep = chunk['pixel_uid'].astype(str).isin(selected_pixel_ids)
    retained = chunk.loc[keep].copy()
    if not retained.empty:
        retained_parts.append(retained)
        retained_count += len(retained)
    print(
        f'Chunk {chunk_index}: scanned={scanned_rows:,}; '
        f'retained={retained_count:,}'
    )
    del chunk, retained
    gc.collect()

if not retained_parts:
    raise ValueError('No locked pixel_uid was found in the source CSV.')
feature_rows = pd.concat(retained_parts, ignore_index=True)
retained_parts.clear()
gc.collect()

if len(feature_rows) != EXPECTED_MODEL_ROWS:
    raise ValueError(
        f'Expected {EXPECTED_MODEL_ROWS:,} feature rows; '
        f'found {len(feature_rows):,}.'
    )
if feature_rows['pixel_uid'].duplicated().any():
    raise ValueError('Recovered predictor rows contain duplicate IDs.')

model_df = training_rows.merge(
    feature_rows,
    on='pixel_uid',
    how='left',
    validate='one_to_one',
)
del feature_rows
gc.collect()

for feature in S2_FEATURES:
    model_df[feature] = pd.to_numeric(
        model_df[feature], errors='coerce'
    ).astype('float32')
model_df[S2_FEATURES] = model_df[S2_FEATURES].replace(
    [np.inf, -np.inf], np.nan
)
if model_df[S2_FEATURES].isna().any(axis=1).any():
    raise ValueError('A locked E2 row has a missing S2 predictor.')

print('E2 modelling table:', model_df.shape)
print('Input rows scanned:', f'{scanned_rows:,}')
print('Input CSV SHA256:', input_csv_hash)
display(model_df.head())


## 6. Define the fixed weighted StandardScaler + R1 RBF-SVC

In every outer fold:

- the scaler is fitted on training rows only using polygon-normalised
  `sample_weight`;
- SVC weights equal polygon weight multiplied by the training-fold class
  frequency factor and are normalised to mean one;
- validation rows are never used in scaler fitting or classifier fitting.


In [ ]:
def make_svm_training_weights(train_frame):
    class_row_counts = train_frame['class_id'].value_counts()
    if set(class_row_counts.index.astype(int)) != set(CLASS_IDS):
        raise ValueError('A training fold is missing a class.')
    class_factors = {
        int(class_id): len(train_frame)
        / (len(CLASS_IDS) * int(row_count))
        for class_id, row_count in class_row_counts.items()
    }
    weights = (
        train_frame['sample_weight'].to_numpy(dtype='float64')
        * train_frame['class_id'].map(class_factors).to_numpy(
            dtype='float64'
        )
    )
    weights *= len(weights) / weights.sum()
    if not np.isclose(weights.mean(), 1.0):
        raise AssertionError('SVC weights are not normalised to mean 1.')
    return weights.astype('float64')


def build_fixed_r1_pipeline():
    classifier = SVC(
        C=10.0,
        kernel='rbf',
        gamma='scale',
        tol=0.001,
        probability=False,
        class_weight=None,
        cache_size=SVC_CACHE_MB,
        shrinking=True,
        decision_function_shape='ovr',
        break_ties=True,
        max_iter=-1,
        random_state=RANDOM_SEED,
    )
    return Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', classifier),
    ])


def fit_fixed_r1(train_frame):
    model = build_fixed_r1_pipeline()
    model.fit(
        train_frame[S2_FEATURES],
        train_frame['class_id'],
        scaler__sample_weight=train_frame[
            'sample_weight'
        ].to_numpy(dtype='float64'),
        classifier__sample_weight=make_svm_training_weights(
            train_frame
        ),
    )
    observed_classes = model.named_steps[
        'classifier'
    ].classes_.astype(int).tolist()
    if observed_classes != CLASS_IDS:
        raise ValueError('Trained SVC class order differs from 0..6.')
    return model


## 7. Define prediction, polygon aggregation and metrics

Polygon predictions are based on the mean of each class decision score
across retained pixels, followed by `argmax`. They are not obtained by
majority vote and do not use calibrated probabilities.


In [ ]:
def make_pixel_predictions(model, validation_frame):
    predicted_ids = model.predict(
        validation_frame[S2_FEATURES]
    ).astype('int64')
    decision_scores = model.decision_function(
        validation_frame[S2_FEATURES]
    )
    model_classes = model.named_steps[
        'classifier'
    ].classes_.astype(int)
    if (
        decision_scores.ndim != 2
        or decision_scores.shape[1] != len(model_classes)
    ):
        raise ValueError('Unexpected multiclass decision-score shape.')

    output = validation_frame[[
        'pixel_uid', 'sample_uid', 'group_uid',
        'class_id', 'fold_id', 'sample_weight',
    ]].copy()
    output['pred_class_id'] = predicted_ids
    for class_id in CLASS_IDS:
        output[f'score_{class_id}'] = np.nan
    for column_index, class_id in enumerate(model_classes):
        output[f'score_{class_id}'] = decision_scores[:, column_index]
    score_columns = [f'score_{class_id}' for class_id in CLASS_IDS]
    if output[score_columns].isna().any().any():
        raise ValueError('At least one decision score is missing.')
    output['feature_set'] = 'S2'
    return output


def aggregate_polygon_predictions(pixel_predictions):
    score_columns = [f'score_{class_id}' for class_id in CLASS_IDS]
    aggregations = {
        'group_uid': 'first',
        'class_id': 'first',
        'fold_id': 'first',
    }
    aggregations.update({column: 'mean' for column in score_columns})
    polygon_predictions = (
        pixel_predictions.groupby('sample_uid', as_index=False)
        .agg(aggregations)
    )
    score_matrix = polygon_predictions[score_columns].to_numpy()
    polygon_predictions['pred_class_id'] = np.asarray(CLASS_IDS)[
        score_matrix.argmax(axis=1)
    ]
    polygon_predictions['feature_set'] = 'S2'
    return polygon_predictions


def metric_dictionary(y_true, y_pred, sample_weight=None):
    precision, recall, class_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=[DURIAN_ID, RUBBER_ID],
            average=None,
            sample_weight=sample_weight,
            zero_division=0,
        )
    )
    return {
        'accuracy': accuracy_score(
            y_true, y_pred, sample_weight=sample_weight
        ),
        'balanced_accuracy': balanced_accuracy_score(
            y_true, y_pred, sample_weight=sample_weight
        ),
        'macro_f1': f1_score(
            y_true,
            y_pred,
            labels=CLASS_IDS,
            average='macro',
            sample_weight=sample_weight,
            zero_division=0,
        ),
        'durian_precision': float(precision[0]),
        'durian_recall': float(recall[0]),
        'durian_f1': float(class_f1[0]),
        'rubber_precision': float(precision[1]),
        'rubber_recall': float(recall[1]),
        'rubber_f1': float(class_f1[1]),
    }


def evaluate_predictions(pixel_predictions, polygon_predictions):
    pixel_metrics = metric_dictionary(
        pixel_predictions['class_id'],
        pixel_predictions['pred_class_id'],
        sample_weight=pixel_predictions['sample_weight'],
    )
    polygon_metrics = metric_dictionary(
        polygon_predictions['class_id'],
        polygon_predictions['pred_class_id'],
    )
    return {
        **{f'pixel_{key}': value for key, value in pixel_metrics.items()},
        **{
            f'polygon_{key}': value
            for key, value in polygon_metrics.items()
        },
    }


## 8. Run fixed R1 across the five locked outer folds

Each polygon and every pixel from that polygon is predicted exactly once
out of fold. Runtime varies by environment and is not part of the locked
scientific result.


In [ ]:
fold_metric_records = []
oof_pixel_parts = []
oof_polygon_parts = []
experiment_start = time.time()

for fold_id in range(N_SPLITS):
    fold_start = time.time()
    train_frame = model_df.loc[model_df['fold_id'] != fold_id].copy()
    validation_frame = model_df.loc[
        model_df['fold_id'] == fold_id
    ].copy()
    if train_frame.empty or validation_frame.empty:
        raise ValueError(f'Fold {fold_id} has an empty partition.')
    if set(train_frame['group_uid']).intersection(
        set(validation_frame['group_uid'])
    ):
        raise ValueError(f'Group leakage detected in fold {fold_id}.')

    model = fit_fixed_r1(train_frame)
    pixel_predictions = make_pixel_predictions(
        model, validation_frame
    )
    polygon_predictions = aggregate_polygon_predictions(
        pixel_predictions
    )
    metrics = evaluate_predictions(
        pixel_predictions, polygon_predictions
    )
    runtime_min = (time.time() - fold_start) / 60
    fold_metric_records.append({
        'feature_set': 'S2',
        'fold_id': fold_id,
        'runtime_min': runtime_min,
        **metrics,
    })
    oof_pixel_parts.append(pixel_predictions)
    oof_polygon_parts.append(polygon_predictions)

    if SAVE_FOLD_MODELS:
        joblib.dump(
            model,
            MODEL_DIR / f'fixed_S2_R1_fold_{fold_id}.joblib',
        )
    print(
        f'Fold {fold_id} complete: '
        f'polygon_accuracy={metrics["polygon_accuracy"]:.6f}; '
        f'runtime={runtime_min:.2f} min'
    )
    del model, train_frame, validation_frame
    gc.collect()

fold_metrics = pd.DataFrame(fold_metric_records)
oof_pixel_predictions = pd.concat(
    oof_pixel_parts, ignore_index=True
)
oof_polygon_predictions = pd.concat(
    oof_polygon_parts, ignore_index=True
)

if len(oof_pixel_predictions) != EXPECTED_MODEL_ROWS:
    raise ValueError('OOF pixel predictions do not cover every row once.')
if oof_pixel_predictions['pixel_uid'].duplicated().any():
    raise ValueError('OOF pixel predictions contain duplicate pixel_uid.')
if len(oof_polygon_predictions) != EXPECTED_SAMPLE_COUNT:
    raise ValueError('OOF polygon predictions do not cover every polygon.')
if oof_polygon_predictions['sample_uid'].duplicated().any():
    raise ValueError('OOF polygon predictions contain duplicate sample_uid.')

print(
    'Total E2 grouped OOF runtime:',
    f'{(time.time() - experiment_start) / 60:.2f} minutes',
)
display(fold_metrics)


## 9. Calculate pooled E2 results and the polygon confusion matrix

Pooled OOF metrics are calculated after concatenating the five validation
folds. Fold means and sample standard deviations (`ddof=1`) are reported
separately.


In [ ]:
pooled_metrics = evaluate_predictions(
    oof_pixel_predictions, oof_polygon_predictions
)

summary_record = {
    'feature_set': 'S2',
    'n_features': len(S2_FEATURES),
    'runtime_min_sum': fold_metrics['runtime_min'].sum(),
    **{f'pooled_{key}': value for key, value in pooled_metrics.items()},
}
for metric_name in [
    'polygon_accuracy',
    'polygon_balanced_accuracy',
    'polygon_macro_f1',
    'polygon_durian_f1',
    'polygon_rubber_f1',
]:
    summary_record[f'{metric_name}_fold_mean'] = fold_metrics[
        metric_name
    ].mean()
    summary_record[f'{metric_name}_fold_sd'] = fold_metrics[
        metric_name
    ].std(ddof=1)

e2_summary = pd.DataFrame([summary_record])
polygon_confusion_array = confusion_matrix(
    oof_polygon_predictions['class_id'],
    oof_polygon_predictions['pred_class_id'],
    labels=CLASS_IDS,
)
polygon_confusion = pd.DataFrame(
    polygon_confusion_array,
    index=REPORTING_CLASS_NAMES,
    columns=REPORTING_CLASS_NAMES,
)
polygon_confusion.index.name = None

display(e2_summary.T)
display(polygon_confusion)


## 10. Verify against the locked E2 benchmark

This is the central reconstruction check. Scientific metrics and the full
7×7 polygon confusion matrix must match the retained E2 evidence. Runtime
is excluded because it depends on hardware and Colab load.


In [ ]:
LOCKED_E2_METRICS = {
    'pooled_pixel_accuracy': 0.7994308298113575,
    'pooled_pixel_balanced_accuracy': 0.7947360904868594,
    'pooled_pixel_macro_f1': 0.7727588701330619,
    'pooled_pixel_durian_precision': 0.7988165077524165,
    'pooled_pixel_durian_recall': 0.8118326566761106,
    'pooled_pixel_durian_f1': 0.8052719884723906,
    'pooled_pixel_rubber_precision': 0.48107290814717124,
    'pooled_pixel_rubber_recall': 0.7599196896467063,
    'pooled_pixel_rubber_f1': 0.5891683410626712,
    'pooled_polygon_accuracy': 0.8545780969479354,
    'pooled_polygon_balanced_accuracy': 0.8483189902467012,
    'pooled_polygon_macro_f1': 0.8315074621606635,
    'pooled_polygon_durian_precision': 0.8412698412698413,
    'pooled_polygon_durian_recall': 0.8833333333333333,
    'pooled_polygon_durian_f1': 0.8617886178861789,
    'pooled_polygon_rubber_precision': 0.6428571428571429,
    'pooled_polygon_rubber_recall': 0.75,
    'pooled_polygon_rubber_f1': 0.6923076923076923,
}
LOCKED_POLYGON_CONFUSION = np.asarray([
    [97, 0, 0, 1, 0, 0, 2],
    [0, 106, 2, 7, 0, 5, 0],
    [1, 7, 65, 4, 0, 1, 0],
    [1, 1, 2, 36, 0, 0, 0],
    [0, 6, 7, 7, 71, 9, 0],
    [0, 5, 3, 0, 1, 27, 0],
    [2, 1, 2, 4, 0, 0, 74],
], dtype='int64')

reproduction_records = []
for metric_name, expected_value in LOCKED_E2_METRICS.items():
    observed_value = float(e2_summary.loc[0, metric_name])
    absolute_difference = abs(observed_value - expected_value)
    reproduction_records.append({
        'check': metric_name,
        'expected': expected_value,
        'observed': observed_value,
        'absolute_difference': absolute_difference,
        'passed': bool(
            np.isclose(
                observed_value,
                expected_value,
                rtol=0,
                atol=RESULT_ABSOLUTE_TOLERANCE,
            )
        ),
    })

confusion_passed = np.array_equal(
    polygon_confusion_array, LOCKED_POLYGON_CONFUSION
)
reproduction_records.append({
    'check': 'polygon_confusion_matrix_exact',
    'expected': 'locked 7x7 integer matrix',
    'observed': 'reproduced 7x7 integer matrix',
    'absolute_difference': int(
        np.abs(
            polygon_confusion_array - LOCKED_POLYGON_CONFUSION
        ).sum()
    ),
    'passed': bool(confusion_passed),
})
reproduction_check = pd.DataFrame(reproduction_records)
all_reproduction_checks_passed = bool(
    reproduction_check['passed'].all()
)

display(reproduction_check)
print('ALL LOCKED E2 CHECKS PASSED:', all_reproduction_checks_passed)
if REQUIRE_LOCKED_RESULT_MATCH and not all_reproduction_checks_passed:
    failed = reproduction_check.loc[
        ~reproduction_check['passed'], 'check'
    ].tolist()
    raise AssertionError(
        'E2 reconstruction differs from locked results: '
        + ', '.join(failed)
    )


## 11. Save auditable E2 outputs

The core filenames match the retained result structure. Class 3 is
written as `Other agriculture` only in the reporting confusion matrix;
numerical encoding remains unchanged.


In [ ]:
pixel_output_columns = [
    'pixel_uid', 'sample_uid', 'group_uid', 'class_id', 'fold_id',
    'sample_weight', 'pred_class_id',
    *[f'score_{class_id}' for class_id in CLASS_IDS],
    'feature_set',
]
polygon_output_columns = [
    'sample_uid', 'group_uid', 'class_id', 'fold_id',
    *[f'score_{class_id}' for class_id in CLASS_IDS],
    'pred_class_id', 'feature_set',
]

output_paths = {
    'oof_pixel_predictions':
        TABLE_DIR / 'fixed_S2_R1_oof_pixel_predictions.csv',
    'oof_polygon_predictions':
        TABLE_DIR / 'fixed_R1_oof_polygon_predictions.csv',
    'fold_metrics': TABLE_DIR / 'fixed_R1_fold_metrics.csv',
    'summary': TABLE_DIR / 'fixed_R1_feature_group_summary.csv',
    'polygon_confusion':
        TABLE_DIR / 'fixed_S2_R1_polygon_confusion_matrix.csv',
    'reproduction_check':
        TABLE_DIR / 'E2_locked_result_reproduction_check.csv',
}

atomic_write_csv(
    oof_pixel_predictions[pixel_output_columns],
    output_paths['oof_pixel_predictions'],
    index=False,
)
atomic_write_csv(
    oof_polygon_predictions[polygon_output_columns],
    output_paths['oof_polygon_predictions'],
    index=False,
)
atomic_write_csv(
    fold_metrics,
    output_paths['fold_metrics'],
    index=False,
)
atomic_write_csv(
    e2_summary,
    output_paths['summary'],
    index=False,
)
atomic_write_csv(
    polygon_confusion,
    output_paths['polygon_confusion'],
    index=True,
)
atomic_write_csv(
    reproduction_check,
    output_paths['reproduction_check'],
    index=False,
)

output_hashes = {
    name: sha256_file(path) for name, path in output_paths.items()
}
run_metadata = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'run_name': RUN_NAME,
    'experiment': 'E2',
    'experiment_definition': (
        'Bentong fixed S2 + R1 five-fold grouped OOF matched benchmark'
    ),
    'analytical_status': 'MATCHED_FIXED_CONFIGURATION_BENCHMARK',
    'does_not_repeat_model_selection': True,
    'rows': len(model_df),
    'polygons': model_df['sample_uid'].nunique(),
    'groups': model_df['group_uid'].nunique(),
    'folds': list(range(N_SPLITS)),
    'features': S2_FEATURES,
    'candidate': 'R1',
    'params': {
        'C': 10.0,
        'kernel': 'rbf',
        'gamma': 'scale',
        'tol': 0.001,
        'probability': False,
        'class_weight': None,
        'cache_size': SVC_CACHE_MB,
        'shrinking': True,
        'decision_function_shape': 'ovr',
        'break_ties': True,
        'max_iter': -1,
        'random_state': RANDOM_SEED,
    },
    'training_weight_method': (
        'polygon sample_weight multiplied by training-fold class '
        'frequency factor, normalised to mean 1'
    ),
    'scaler_weight_method': (
        'polygon-normalised sample_weight; fitted within each outer '
        'training fold'
    ),
    'polygon_aggregation': 'mean multiclass decision score',
    'internal_class_to_id': CLASS_TO_ID,
    'reporting_id_to_class': ID_TO_REPORTING_CLASS,
    'class_3_note': (
        'Internal label Mixed agriculture is reported as Other '
        'agriculture; class_id remains 3.'
    ),
    'input_csv': str(INPUT_CSV),
    'input_csv_sha256': input_csv_hash,
    'training_rows': str(TRAINING_ROWS_PATH),
    'training_rows_sha256': training_rows_hash,
    'fold_assignments': str(FOLD_ASSIGNMENTS_PATH),
    'fold_assignments_sha256': fold_assignments_hash,
    'software': {
        'python': platform.python_version(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__,
    },
    'all_locked_e2_checks_passed': all_reproduction_checks_passed,
    'output_sha256': output_hashes,
}
metadata_path = METADATA_DIR / 'run_metadata.json'
atomic_write_json(run_metadata, metadata_path)

readme_text = (
    'E2 Bentong fixed S2 + R1 grouped OOF benchmark\n'
    '==================================================\n\n'
    'This directory was generated by the standalone E2 reconstruction '
    'notebook.\nE2 reuses the exact 37,724 retained Bentong rows and '
    'original five outer folds.\nIt fixes the 13 S2 predictors and R1 '
    'RBF-SVC configuration; no feature selection\nor hyperparameter '
    'tuning is performed. Polygon predictions use mean multiclass\n'
    'decision scores. Class 3 is reported as Other agriculture without '
    'changing its\nnumeric encoding. Inspect tables/'
    'E2_locked_result_reproduction_check.csv before\nusing the '
    'reconstructed outputs. E2 is a matched source-domain benchmark '
    'for E3,\nnot an independent target-domain evaluation and not a '
    'replacement for E1.\n'
)
readme_path = OUTPUT_DIR / 'README_E2.txt'
temporary_readme = readme_path.with_name(readme_path.name + '.tmp')
temporary_readme.write_text(readme_text, encoding='utf-8')
os.replace(temporary_readme, readme_path)

print('Saved E2 output directory:', OUTPUT_DIR)
print('Metadata:', metadata_path)
print('README:', readme_path)
display(pd.DataFrame([
    {'output': name, 'path': str(path), 'sha256': output_hashes[name]}
    for name, path in output_paths.items()
]))


## 12. Final completion gate

E2 reconstruction is complete only when all structural audits and all
locked-result comparisons pass. The expected headline polygon result is
OA `0.8545780969479354`, macro-F1 `0.8315074621606635`, Durian F1
`0.8617886178861789`, and Rubber F1 `0.6923076923076923`.


In [ ]:
completion_status = {
    'experiment': 'E2',
    'rows_complete': len(oof_pixel_predictions) == EXPECTED_MODEL_ROWS,
    'polygons_complete': (
        len(oof_polygon_predictions) == EXPECTED_SAMPLE_COUNT
    ),
    'groups_locked': model_df['group_uid'].nunique() == EXPECTED_GROUP_COUNT,
    'folds_locked': sorted(model_df['fold_id'].unique()) == list(
        range(N_SPLITS)
    ),
    'features_locked': S2_FEATURES,
    'candidate_locked': R1_PARAMS,
    'all_locked_result_checks_passed': all_reproduction_checks_passed,
    'ready_for_thesis_E2_use': bool(
        len(oof_pixel_predictions) == EXPECTED_MODEL_ROWS
        and len(oof_polygon_predictions) == EXPECTED_SAMPLE_COUNT
        and all_reproduction_checks_passed
    ),
}
print(json.dumps(completion_status, indent=2, ensure_ascii=False))
if not completion_status['ready_for_thesis_E2_use']:
    raise AssertionError('E2 completion gate did not pass.')
